# Etapa 6: Feature Engineering
---

In [1]:
# Imports
import sys
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# Obtener ruta raíz y agregarla a sys.path para poder importar desde src/.
PROJECT_ROOT = Path(__file__).resolve().parent
sys.path.append(str(PROJECT_ROOT))   

# Ruta del dataset procesado
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "bank_marketing.csv"
df = pd.read_csv(PROCESSED_PATH)

---
## Repaso del decisiones


### Codificación

Para codificar las categorías de cada una de las variables predictoras se usará One-Hot Encoding. La razón principal es porque, mientras los modelos de KNN y RL se basen en distancias, al usar otro método como Label Encoding, interpretarían que una categoría está más lejana que otra; con One-Hot Encoding evita esos errores al crear una columna booleana indicando a cual categoría pertenece la instancia, marcandola con un `1` y a las que no pertenece con `0`.

En cuanto a la conocida "maldición de la dimensionalidad" en machine learning, no es un problema en este caso, ya codificadas el total de columnas sería aproximadament 50. Contemplando que se cuenta con ~45,000 instancias, tener 50 columnas no será un problema en el entrenamiento.

- La variable target `y` se excluye de dicha codificación, simplemente se codifica como 0 si `no` y 1 si `yes`.

- Se utiliza `handle_unknown="ignore"` para que el pipeline pueda procesar categorías que no hayan aparecido durante el entrenamiento, favoreciendo su reutilización en producción.

### Escalado

El escalado aplica unicamente para el modelo de KNN y el de Regresión Logística. Esto debido a que ambos modelos requieren escalado a diferencia de los árboles. Por un lado porque KNN funciona por distancias, por ende, si se mantienen rangos muy distintos como es el caso de `balance` y `age`, `age` quedará casi completamente dominado por la diferencia de `balance`; mientras que Regresión Logistica lo requiere porque se entrena con regularización y esta va a penalizar de forma desigual a las escalas grandes como en el caso de `balance`. Por otro lado, en Árbol de decisión y Random Forest, las variables numéricas se mantienen en su escala original, sin aplicar escalado ni transformaciones por asimetría, ya que los modelos basados en árboles no dependen de la escala de las variables ni requieren que estas presenten distribuciones normales.

Se considera ideal usar el escalado RobustScaler. La decisión se justifica en base a que los otros métodos más conocidos son bastante sensibles a outliers, y durante el diagnóstico se identificaron casos de outliers y de asimetría en algunas de las variables numéricas que estas se consideraron casos posibles del negocio.

### Balanceo de clases

El dataset presenta un desbalance considerable en la variable objetivo (88.3% `no` / 11.7% `yes`), identificado durante el EDA. Para tratarlo, se decidió una estrategia distinta según el modelo, ya que no todos soportan las mismas técnicas.

- Para Árbol de Decisión, Random Forest y Regresión Logística se utilizará `class_weight="balanced"`, disponible de forma nativa en los tres. Esta opción no modifica los datos, sino que penaliza más los errores en la clase minoritaria durante el entrenamiento, sin necesidad de generar registros adicionales. Su efectividad será posteriormente evaluada mediante los experimentos registrados en MLflow, utilizando métricas como Precision, Recall, F1 y AUC.

- Para KNN se utilizará SMOTE, ya que este modelo no cuenta con un parámetro equivalente a `class_weight`. SMOTE genera registros sintéticos de la clase minoritaria interpolando entre vecinos existentes, aplicado únicamente sobre el conjunto de entrenamiento y después del split, para evitar fuga de información hacia el conjunto de prueba.

Se es consciente de que esto puede generar columnas categóricas con valores intermedios no interpretables directamente (por ejemplo, un valor de 0.35 en una columna binaria de One-Hot Encoding). El impacto real de esta decisión se evaluará de forma empírica durante la etapa de tracking, comparando el desempeño de KNN con y sin SMOTE mediante las métricas correspondientes como Accuracy, F1, Recall, en lugar de asumirlo de antemano.


### Decisión sobre `pdays`

La variable `pdays` se conserva en su forma original, incluyendo el valor `-1`, debido a que este valor representa que el cliente no había sido contactado previamente y, por lo tanto, contiene información útil para el modelo. Tampoco se crea una variable derivada como `contactado_previamente`, ya que esta información puede obtenerse a partir de las variables existentes, principalmente `pdays` y `previous`, por lo que crear una nueva variable podría introducir redundancia sin aportar información adicional relevante.

* Las transformaciones mencionadas anteriormente se encuentran encapsuladas en un pipeline reutilizable para garantizar que el mismo proceso pueda aplicarse durante el entrenamiento y posteriormente en producción.


---

In [3]:
df.sample(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
18318,39,services,divorced,secondary,no,42,no,no,cellular,31,jul,75,2,-1,0,unknown,no
23525,40,management,married,tertiary,no,1666,yes,no,cellular,28,aug,271,4,-1,0,unknown,no
42531,39,unemployed,single,secondary,no,804,no,no,cellular,21,dec,193,8,203,1,success,no
26947,35,unemployed,divorced,tertiary,no,8903,yes,no,cellular,21,nov,336,6,-1,0,unknown,no
19075,40,management,married,secondary,no,1251,yes,no,cellular,5,aug,406,1,-1,0,unknown,no


In [4]:
from src.features.build_features import build_pipeline, encode_target, feature_selection

In [5]:
# Separación de las variables predictoras (X) de la variable objetivo (y)
# feature_selection se encarga de eliminar 'duration' y 'y' en X
X = feature_selection(df)
y = encode_target(df["y"])

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

# Verificar las variables que serán utilizadas como predictoras
print("Variables predictoras:")
print(X.columns.tolist())

Dimensiones de X: (45211, 15)
Dimensiones de y: (45211,)
Variables predictoras:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


In [6]:
# Para validar que los pipelines funcionan
from src.pipelines.split import split_data

X_train, X_test, y_train, y_test = split_data(X, y)

In [7]:
# Comparamos la distribución de la variable objetivo entre entrenamiento y prueba
# para comprobar que el desbalance de clases se mantiene aproximadamente igual.

print("Distribución en entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribución en prueba:")
print(y_test.value_counts(normalize=True).round(3))

Distribución en entrenamiento:
y
0    0.883
1    0.117
Name: proportion, dtype: float64

Distribución en prueba:
y
0    0.883
1    0.117
Name: proportion, dtype: float64


In [8]:
# Pipeline de Decision Tree: 
# Codifica las categóricas y aplica class_weight="balanced" para compensar el desbalance de clases (88.3% no / 11.7% yes).

decision_tree_pipeline = build_pipeline(
    model=DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=4),
    incluir_escalado=False
)

print("Pipeline de Decision Tree creado correctamente.")

Pipeline de Decision Tree creado correctamente.


In [9]:
# Creamos el pipeline de Random Forest.
# Codifica las categóricas y aplica class_weight="balanced" para compensar el desbalance de clases (88.3% no / 11.7% yes).

random_forest_pipeline = build_pipeline(
    model=RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=100, max_depth=5),
    incluir_escalado=False
)

print("Pipeline de Random Forest creado correctamente.")

Pipeline de Random Forest creado correctamente.


In [10]:
k = 3  # Valor provisional para validar el pipeline; el valor final se determina en la etapa de tracking (MLflow)

# Pipeline de KNN: 
# Codifica categóricas, escala numéricas con RobustScaler (KNN es sensible a la escala), 
# y aplica SMOTE sobre el train para tratar el desbalance, ya que KNN no soporta class_weight.

knn_pipeline = build_pipeline(
    model=KNeighborsClassifier(n_neighbors=k),
    incluir_escalado=True,
    incluir_smote=True  
)

print("Pipeline de KNN creado correctamente.")

Pipeline de KNN creado correctamente.


In [11]:
# Pipeline de Regresión Logística: 
# Codifica las categóricas con One-Hot y escala las numéricas con RobustScaler (sensible a la escala por la regularización).
# Se usa class_weight="balanced" para compensar el desbalance de clases (88.3% no / 11.7% yes).

rl_pipeline = build_pipeline(
    model=LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000),
    incluir_escalado=True
)

print("Pipeline de Regresión Logística creado correctamente.")

Pipeline de Regresión Logística creado correctamente.


----

In [12]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

resultados = []
for k in [3, 5, 7, 9, 11, 15]:
    knn_pipeline = build_pipeline(
        model=KNeighborsClassifier(n_neighbors=k),
        incluir_escalado=True,
        incluir_smote=True
    )
    scores = cross_val_score(
        knn_pipeline, X_train, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring="average_precision"
    )
    resultados.append((k, scores.mean()))

resultados

[(3, np.float64(0.20625521822649245)),
 (5, np.float64(0.22578461483293247)),
 (7, np.float64(0.23963758658708234)),
 (9, np.float64(0.25347757650878255)),
 (11, np.float64(0.264489159959863)),
 (15, np.float64(0.28602134996986595))]

In [13]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

resultados = []
for k in [15, 17, 19, 21, 25, 31, 33]:
    knn_pipeline = build_pipeline(
        model=KNeighborsClassifier(n_neighbors=k),
        incluir_escalado=True,
        incluir_smote=True
    )
    scores = cross_val_score(
        knn_pipeline, X_train, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring="average_precision"
    )
    resultados.append((k, scores.mean()))

resultados

[(15, np.float64(0.28602134996986595)),
 (17, np.float64(0.29277461257969317)),
 (19, np.float64(0.29885537664464007)),
 (21, np.float64(0.30366103311182097)),
 (25, np.float64(0.3136876160504028)),
 (31, np.float64(0.32117267871891625)),
 (33, np.float64(0.32407398661317605))]

### Decisión del valor de k en KNN

Se evaluaron diferentes valores de k mediante una curva de validación (variación de k con validación cruzada estratificada), utilizando únicamente valores impares para evitar empates en la votación de clases entre vecinos.

* Inicialmente se evaluaron valores bajos, de 3 a 15, observando que el score seguía aumentando de forma considerable de un valor a otro, sin señales de estabilizarse.

* Se amplió el rango de 15 a 33 (con algunos saltos), observando que el aumento entre valores consecutivos se hizo más moderado, pero sin encontrarse un techo claro.

* No se probaron valores mayores a 33 por consideraciones prácticas: un k muy alto aumenta el costo computacional y reduce la interpretabilidad del modelo (la predicción deja de basarse en un vecindario realmente "cercano"). Se decidió trabajar con dos configuraciones para las corridas formales: k=15, como opción más conservadora y liviana, y k=31, que mostró mejor score en la validación cruzada, para contrastar ambos extremos del trade-off entre desempeño y costo/interpretabilidad.